In [96]:
import requests
import pandas as pd
from datetime import datetime, timezone
from supabase import create_client, Client


def get_btc_weekly_data():
    """Extracting the 200-week data of Bitcoin from Yahoo Finance"""
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/BTC-USD"
    params = {
        "period1": 0,
        "period2": int(datetime.now(timezone.utc).timestamp()),
        "interval": "1wk",
        "events": "history",
    }

    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )
    response.raise_for_status()
    data = response.json()["chart"]["result"][0]

    df = pd.DataFrame({
        "Date": pd.to_datetime(data["timestamp"], unit="s", utc=True),
        "Close": data["indicators"]["quote"][0]["close"]
    })
    return df


def calculate_200_wma():
    """Calculate the 200-week moving average BTC price"""
    df_btc = get_btc_weekly_data()

    # Calculate 200-week moving average
    df_btc["200WMA"] = df_btc["Close"].rolling(
        window=200,
        min_periods=200
    ).mean()

    result = (df_btc.dropna(subset=["200WMA"])[["Date", "200WMA"]]).tail(1)

    return result


def supabase_connection():

    SUPABASE_URL = "https://mzhomnjkelxmwlhafisp.supabase.co"
    SUPABASE_KEY = "sb_publishable_fELh3_L2GsNTRbM-Pzdaxg_Xfk5bXXL"

    # Establishing a supabase connection
    supabase: Client = create_client(
        SUPABASE_URL,
        SUPABASE_KEY
    )

    return supabase


def supabase_data_import():

    row = calculate_200_wma().iloc[0].to_dict()
    row["Date"] = row["Date"].isoformat()

    response = (
        supabase_connection()
        .table("200_wma_btc")
        .insert(row)
        .execute()
    )

    return response

def return_supabase_wma():
    response = (
        supabase
        .table("200_wma_btc")
        .select("*")
        .execute()
        )

    df = pd.DataFrame(response.data).sort_values(by="Date", ascending=False).head(1)

    return df

return_supabase_wma()["200WMA"].reset_index(drop=True)

0    64219.20043
Name: 200WMA, dtype: float64

In [85]:
# Establishing a supabase connection
supabase: Client = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)

response = (
    supabase
    .table("200_wma_btc")
    .select("*")
    .execute()
)

df = pd.DataFrame(response.data).sort_values(by="Date", ascending=False)
df.head(5)

,Date,200WMA,ID
200,2026-08-12T16:34:32+00:00,64219.200430,202
199,2026-08-12T08:05:28+00:00,64221.888418,200
202,2026-08-11T16:58:15+00:00,64219.038477,204
201,2026-08-11T16:58:05+00:00,64219.055020,203
198,2026-08-11T00:00:00+00:00,63999.628066,199
